# 单个 TSFM 的细粒度误差分析

本 notebook 会：
1. 加载某个 TSFM 在所有 correction windows 上的详细预测记录。
2. 提取历史窗口画像特征（统计 + 简易分解 + 频域）。
3. 构建**特征桶组合 -> 误差统计**热力图（均值/最大值/标准差）。
4. 构建**误差桶 -> 特征统计**热力图（均值/最大值/标准差）。

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from analysis.tsfm_error_analysis import (
    list_tsfms,
    load_tsfm_window_dataframe,
    add_quantile_buckets,
    compute_feature_combo_error_stats,
    plot_feature_pair_heatmaps,
    build_error_interval_feature_stats,
    plot_error_bucket_feature_heatmap,
)

plt.rcParams['figure.dpi'] = 130

In [ ]:
# ====== 配置 ======
OUTPUT_ROOT = 'correction_datasets_double_res_0'
TSFM_NAME = None  # 例如 'chronos_large'

all_tsfms = list_tsfms(OUTPUT_ROOT)
print('可用 TSFM:', all_tsfms)
if TSFM_NAME is None and all_tsfms:
    TSFM_NAME = all_tsfms[0]
print('当前分析 TSFM =', TSFM_NAME)

In [ ]:
df = load_tsfm_window_dataframe(OUTPUT_ROOT, TSFM_NAME)
print('样本数:', len(df))
display(df.head(3))

In [ ]:
# ====== 选择画像特征 ======
feature_cols = [
    'hist_std', 'trend_slope', 'trend_strength',
    'fft_low_ratio', 'fft_high_ratio', 'fft_dominant_freq'
]
error_metric = 'mae'

# 为特征构建分位数桶
df_b = add_quantile_buckets(df, feature_cols=feature_cols, n_bins=5)

# 示例：二维桶组合（trend_strength x fft_high_ratio）
stats = compute_feature_combo_error_stats(
    df_b,
    bucket_features=['trend_strength', 'fft_high_ratio'],
    error_metric=error_metric,
)
display(stats.head(10))

In [ ]:
# 热力图：同一特征桶组合下误差均值/最大值/标准差
fig = plot_feature_pair_heatmaps(stats, 'trend_strength_bucket', 'fft_high_ratio_bucket')
plt.show()

In [ ]:
# 误差分桶后，观察各特征在不同误差区间的分布统计
err_feat_stats = build_error_interval_feature_stats(
    df,
    error_metric=error_metric,
    feature_cols=feature_cols,
    n_error_bins=6,
)
display(err_feat_stats)

In [ ]:
for stat in ['mean', 'max', 'std']:
    fig = plot_error_bucket_feature_heatmap(err_feat_stats, feature_cols, stat=stat, figsize=(12, 5))
    plt.show()